# 07 — Model Evaluation

Compares three Bible reference extractors on a held-out annotated test set.

| Model | Class | Role |
|---|---|---|
| **Rule-based (Regex)** | `BibleReferenceAnnotator` | **Baseline** |
| **CRF** | `CRFBibleReferenceExtractor` | Sequence labeller |
| **NER (IndoBERT)** | `BibleReferenceExtractor` | Fine-tuned transformer |

### Metrics
Two evaluation modes are applied at **structure level** (parsed fields):

* **Exact match** — predicted reference must be identical to the gold annotation.
* **Partial match** — predicted and gold spans must overlap by ≥ 50 % of the shorter span (span level), or at least `book_start` and `start_chapter` must agree (structure level).

For each mode: **Precision**, **Recall**, **F1**.

## 1 · Imports & Paths

In [4]:
import json
import sys
import warnings
import numpy as np
from pathlib import Path
from typing import Any, Dict, List


import pandas as pd

warnings.filterwarnings('ignore')

root = Path('..').resolve()
sys.path.append(str(root / 'src'))

from bible_data import load_bible_data
from config.settings import DATA_DIR, MODEL_DIR, MODEL_PATH, CONFIG_PATH

CRF_MODEL_PATH = MODEL_DIR / 'crf-bible-ner.pkl'
W2V_MODEL_PATH = MODEL_DIR / 'word2vec-bible.model'

print('Setup complete.')

Setup complete.


## 2 · Load Data

In [2]:
bible_data = load_bible_data()
print(f'Loaded {len(bible_data['books'])} books.')

Loaded 66 books.


In [6]:
with open(DATA_DIR / 'gold_data.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

df_gold = pd.DataFrame(data)[['text', 'spans', 'structures']]
print(f'Test set: {len(df_gold)} messages, '
      f'{df_gold['structures'].apply(len).sum()} total gold references')
df_gold.head()

Test set: 60 messages, 93 total gold references


,text,spans,structures
0,Efesus 1-2 done,"[{'start': 0, 'end': 10, 'label': 'BIBLE_REF',...","[{'book_start': 'Efesus', 'start_chapter': 1, ..."
1,Why 1-2 done,"[{'start': 0, 'end': 7, 'label': 'BIBLE_REF', ...","[{'book_start': 'Why', 'start_chapter': 1, 'bo..."
2,Efs1-2 done,"[{'start': 0, 'end': 6, 'label': 'BIBLE_REF', ...","[{'book_start': 'Efs', 'start_chapter': 1, 'bo..."
3,_Bil 1-2_ ✓,"[{'start': 1, 'end': 8, 'label': 'BIBLE_REF', ...","[{'book_start': 'Bil', 'start_chapter': 1, 'bo..."
4,Gal 3-6 done,"[{'start': 0, 'end': 7, 'label': 'BIBLE_REF', ...","[{'book_start': 'Gal', 'start_chapter': 3, 'bo..."


In [7]:
def span_exact_match(pred: Dict, gold: Dict) -> bool:
    """True if predicted span has identical start, end, label."""
    return (pred['start'] == gold['start']
            and pred['end'] == gold['end']
            and pred.get('label') == gold.get('label'))

def span_overlap_ratio(pred: Dict, gold: Dict) -> float:
    """Intersection / length-of-shorter span."""
    lo = max(pred['start'], gold['start'])
    hi = min(pred['end'], gold['end'])
    intersection = max(0, hi - lo)
    shorter = min(pred['end'] - pred['start'], gold['end'] - gold['start'])
    return intersection / shorter if shorter > 0 else 0.0

def span_partial_match(pred: Dict, gold: Dict,threshold: float = 0.5) -> bool:
    return span_overlap_ratio(pred, gold) >= threshold

def struct_exact_match(pred: Dict, gold: Dict) -> bool:
    """All four fields must be identical."""
    return (
        (pred.get('book_start') or '').lower() == (gold.get('book_start') or '').lower()
        and pred.get('start_chapter') == gold.get('start_chapter')
        and (pred.get('book_end') or '').lower() == (gold.get('book_end') or '').lower()
        and pred.get('end_chapter') == gold.get('end_chapter')
    )

def compute_prf(
    all_preds: List[List[Dict]],
    all_golds: List[List[Dict]],
    match_fn,
) -> Dict[str, float]:
    """Micro-averaged P / R / F1 using a greedy one-to-one matching strategy."""
    tp = fp = fn = 0

    for preds, golds in zip(all_preds, all_golds):
        matched_gold = set()
        matched_pred = set()

        # Greedy matching: each pred can match at most one gold
        for pi, pred in enumerate(preds):
            for gi, gold in enumerate(golds):
                if gi in matched_gold:
                    continue
                if match_fn(pred, gold):
                    tp += 1
                    matched_gold.add(gi)
                    matched_pred.add(pi)
                    break

        fp += len(preds) - len(matched_pred)
        fn += len(golds) - len(matched_gold)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0)

    return {'precision': precision, 'recall': recall, 'f1': f1,
            'tp': tp, 'fp': fp, 'fn': fn}


def evaluate_model(
    name: str,
    pred_spans,
    pred_structs,
    gold_spans,
    gold_structs,
) -> Dict[str, Any]:
    """Run all four metric combinations for a single model."""
    return {
        'model': name,
        'span_exact': compute_prf(pred_spans, gold_spans, span_exact_match),
        'span_partial': compute_prf(pred_spans, gold_spans, span_partial_match),
        'struct_exact': compute_prf(pred_structs, gold_structs, struct_exact_match),
    }


print('Helpers defined.')

Helpers defined.


## 5 · Model 1 — Rule-Based Regex (Baseline)

In [9]:
from extraction.rule_based import BibleRegexBuilder, RuleBasedExtractor
from services import preprocess

regex_builder = BibleRegexBuilder(bible_data)
regex_parser = RuleBasedExtractor(regex_builder.patterns)

regex_spans = []
regex_structs = []

for row in data:
    text = preprocess(row['text'])
    spans = regex_parser.extract_ner_spans(text)
    structs_raw = regex_parser.extract_structure(text)
    regex_spans.append(spans)
    regex_structs.append(structs_raw)

print(f'Regex extracted spans per message: '
      f'{[len(s) for s in regex_spans]}')

Regex extracted spans per message: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 10, 3, 2, 2, 1, 1, 1, 1, 1, 1, 3, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 2]


In [10]:
crf_spans = []
crf_structs = []
CRF_AVAILABLE = CRF_MODEL_PATH.exists() and W2V_MODEL_PATH.exists()

if CRF_AVAILABLE:
    from extraction.crf_extractor import CRFBibleReferenceExtractor

    crf_extractor = CRFBibleReferenceExtractor(
        crf_path=CRF_MODEL_PATH,
        w2v_path=W2V_MODEL_PATH
    )

    for row in data:
        text = preprocess(row['text'])
        spans = crf_extractor.extract_ner_spans(text)
        structs_raw = crf_extractor.extract_structure(text)
        crf_spans.append(spans)
        crf_structs.append(structs_raw)

    print(f'CRF spans per message: {[len(s) for s in crf_spans]}')
else:
    print(f'CRF artefacts not found at {CRF_MODEL_PATH} — '
          'filling with empty predictions (all zeros).')
    crf_spans = [[] for _ in data]
    crf_structs = [[] for _ in data]


2026-04-22 10:41:48 | INFO     | bible_pipeline.extraction.crf_extractor | CRFBibleReferenceExtractor ready — model: crf-bible-ner.pkl | W2V vocab: 1462
CRF spans per message: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 2, 2, 1, 1, 1, 1, 1, 0, 3, 2, 2, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 0, 1, 1, 2]


In [11]:
ner_spans = []
ner_structs = []
NER_AVAILABLE = MODEL_PATH.exists()

if NER_AVAILABLE:
    from extraction.extractor import BibleReferenceExtractor

    ner_extractor = BibleReferenceExtractor(
        saved_path=MODEL_PATH,
        config_path=CONFIG_PATH,
    )

    # Batch inference for efficiency
    texts = [preprocess(row['text']) for row in data]
    ner_spans = ner_extractor.extract_batch(texts, return_spans=True)
    ner_structs = ner_extractor.extract_batch(texts, return_spans=False)

    print(f'NER spans per message: {[len(s) for s in ner_spans]}')
else:
    print(f'NER checkpoint not found at {MODEL_PATH} — '
          'filling with empty predictions (all zeros).')
    ner_spans = [[] for _ in data]
    ner_structs = [[] for _ in data]

2026-04-22 10:42:01 | INFO     | bible_pipeline.extraction.extractor | Loading model from saved path: C:\one one\Desktop\bible_reading_recap_nlp\models\indobert-bible-ner-v4


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

NER spans per message: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 10, 3, 2, 2, 1, 1, 1, 1, 1, 1, 3, 2, 3, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 2]


In [12]:
gold_spans = [row['spans'] for row in data]
gold_structs = [row['structures'] for row in data]

print(f'Total gold spans     : {sum(len(s) for s in gold_spans)}')
print(f'Total gold structures: {sum(len(s) for s in gold_structs)}')

Total gold spans     : 79
Total gold structures: 93


In [13]:
results = [
    evaluate_model('Regex (baseline)', regex_spans,  regex_structs, gold_spans, gold_structs),
    evaluate_model('CRF', crf_spans, crf_structs, gold_spans, gold_structs),
    evaluate_model('IndoBERT NER', ner_spans, ner_structs, gold_spans, gold_structs),
]

rows = []
METRIC_KEYS = ['span_exact', 'span_partial', 'struct_exact']
METRIC_LABELS = {
    'span_exact': 'Span — Exact',
    'span_partial': 'Span — Partial',
    'struct_exact': 'Structure — Exact',
}

for res in results:
    for mk in METRIC_KEYS:
        m = res[mk]
        rows.append({
            'Model': res['model'],
            'Metric': METRIC_LABELS[mk],
            'Precision': round(m['precision'], 4),
            'Recall': round(m['recall'], 4),
            'F1': round(m['f1'], 4),
            'TP': m['tp'],
            'FP': m['fp'],
            'FN': m['fn'],
        })

df_eval = pd.DataFrame(rows)
df_eval


,Model,Metric,Precision,Recall,F1,TP,FP,FN
0,Regex (baseline),Span — Exact,0.8333,0.8228,0.8280,65,13,14
1,Regex (baseline),Span — Partial,0.9744,0.9620,0.9682,76,2,3
2,Regex (baseline),Structure — Exact,0.8718,0.7312,0.7953,68,10,25
3,CRF,Span — Exact,0.9242,0.7722,0.8414,61,5,18
4,CRF,Span — Partial,0.9848,0.8228,0.8966,65,1,14
5,CRF,Structure — Exact,0.9512,0.8387,0.8914,78,4,15
6,IndoBERT NER,Span — Exact,0.9747,0.9747,0.9747,77,2,2
7,IndoBERT NER,Span — Partial,1.0000,1.0000,1.0000,79,0,0
8,IndoBERT NER,Structure — Exact,0.9785,0.9785,0.9785,91,2,2


In [14]:
def error_analysis(name, pred_structs, gold_structs, gold_data, match_fn):
    print(f"{'='*60}")
    print(f" {name}")
    print(f"{'='*60}")

    for i, (preds, golds, row) in enumerate(zip(pred_structs, gold_structs, gold_data)):
        matched_gold = set()
        matched_pred = set()

        for pi, pred in enumerate(preds):
            for gi, gold in enumerate(golds):
                if gi in matched_gold:
                    continue
                if match_fn(pred, gold):
                    matched_gold.add(gi)
                    matched_pred.add(pi)
                    break
        
        fps = [preds[pi] for pi in range(len(preds)) if pi not in matched_pred]
        fns = [golds[gi] for gi in range(len(golds)) if gi not in matched_gold]

        if fps or fns:
            print(f"\n[{i}] \"{row['text']}\"")
            for fp in fps:
                print(f"  FP (wrong pred) : {fp}")
            for fn in fns:
                print(f"  FN (missed gold): {fn}")

In [15]:
# span error analysis
error_analysis('Regex (baseline)', regex_spans, gold_spans, data, span_exact_match)
error_analysis('CRF', crf_spans, gold_spans, data, span_exact_match)
error_analysis('IndoBERT NER', ner_spans, gold_spans, data, span_exact_match)

 Regex (baseline)

[12] "Ibrani 11-Yakobus 1; 2-3; 4-5"
  FP (wrong pred) : {'start': 0, 'end': 19, 'label': 'BIBLE_REF', 'text': 'Ibrani 11-Yakobus 1'}
  FN (missed gold): {'start': 0, 'end': 29, 'label': 'BIBLE_REF', 'text': 'Ibrani 11-Yakobus 1; 2-3; 4-5'}

[15] "1 Kor 1,2,3,4,5 done"
  FP (wrong pred) : {'start': 0, 'end': 7, 'label': 'BIBLE_REF', 'text': '1 Kor 1'}
  FN (missed gold): {'start': 0, 'end': 15, 'label': 'BIBLE_REF', 'text': '1 Kor 1,2,3,4,5'}

[16] "Mazmur 1, 23, 91 selesai"
  FP (wrong pred) : {'start': 0, 'end': 8, 'label': 'BIBLE_REF', 'text': 'Mazmur 1'}
  FN (missed gold): {'start': 0, 'end': 16, 'label': 'BIBLE_REF', 'text': 'Mazmur 1, 23, 91'}

[17] "Mat 5, 6, 7 done"
  FP (wrong pred) : {'start': 0, 'end': 5, 'label': 'BIBLE_REF', 'text': 'Mat 5'}
  FN (missed gold): {'start': 0, 'end': 11, 'label': 'BIBLE_REF', 'text': 'Mat 5, 6, 7'}

[18] "Kej 1,3,5,7 selesai"
  FP (wrong pred) : {'start': 0, 'end': 5, 'label': 'BIBLE_REF', 'text': 'Kej 1'}
  FN (missed gol

In [16]:
# structural error analysis
error_analysis('Regex (baseline)', regex_structs, gold_structs, data, struct_exact_match)
error_analysis('CRF', crf_structs, gold_structs, data, struct_exact_match)
error_analysis('IndoBERT NER', ner_structs, gold_structs, data, struct_exact_match)

 Regex (baseline)

[12] "Ibrani 11-Yakobus 1; 2-3; 4-5"
  FN (missed gold): {'book_start': 'Yakobus', 'start_chapter': 2, 'book_end': None, 'end_chapter': 3}
  FN (missed gold): {'book_start': 'Yakobus', 'start_chapter': 4, 'book_end': None, 'end_chapter': 5}

[15] "1 Kor 1,2,3,4,5 done"
  FN (missed gold): {'book_start': '1 Kor', 'start_chapter': 2, 'book_end': None, 'end_chapter': None}
  FN (missed gold): {'book_start': '1 Kor', 'start_chapter': 3, 'book_end': None, 'end_chapter': None}
  FN (missed gold): {'book_start': '1 Kor', 'start_chapter': 4, 'book_end': None, 'end_chapter': None}
  FN (missed gold): {'book_start': '1 Kor', 'start_chapter': 5, 'book_end': None, 'end_chapter': None}

[16] "Mazmur 1, 23, 91 selesai"
  FN (missed gold): {'book_start': 'Mazmur', 'start_chapter': 23, 'book_end': None, 'end_chapter': None}
  FN (missed gold): {'book_start': 'Mazmur', 'start_chapter': 91, 'book_end': None, 'end_chapter': None}

[17] "Mat 5, 6, 7 done"
  FN (missed gold): {'book_star

In [17]:
def bootstrap_ci(pred_structs, gold_structs, match_fn, n_iterations=1000, ci=0.95):
    indices = list(range(len(gold_structs)))
    f1_scores = []

    for _ in range(n_iterations):
        sample = np.random.choice(indices, size=len(indices), replace=True)
        sampled_preds = [pred_structs[i] for i in sample]
        sampled_golds = [gold_structs[i] for i in sample]

        result = compute_prf(sampled_preds, sampled_golds, match_fn)
        f1_scores.append(result['f1'])

    alpha = 1 - ci
    lower = np.percentile(f1_scores, 100 * (alpha / 2))
    upper = np.percentile(f1_scores, 100 * (1 - alpha / 2))
    return lower, upper

models = [
    ('Regex (baseline)', regex_spans, regex_structs),
    ('CRF', crf_spans, crf_structs),
    ('IndoBERT NER', ner_spans, ner_structs)
]

np.random.seed(42)
print(f"{'Model':<20} {'Metric':<20} {'F1':>6}  {'95% CI'}")
print('-' * 60)
for name, pred_spans, pred_structs in models:
    for label, match_fn, preds, golds in [
        ('Spans Exact', span_exact_match, pred_spans, gold_spans), 
        ('Spans Partial', span_partial_match, pred_spans, gold_spans), 
        ('Structural', struct_exact_match, pred_structs, gold_structs),
    ]:
        result = compute_prf(preds, golds, match_fn)
        lo, hi = bootstrap_ci(preds, golds, match_fn)
        print(f"{name:<20} {label:<20} {result['f1']:.4f}  [{lo:.4f}, {hi:.4f}]")

Model                Metric                   F1  95% CI
------------------------------------------------------------
Regex (baseline)     Spans Exact          0.8280  [0.7237, 0.9067]
Regex (baseline)     Spans Partial        0.9682  [0.9242, 0.9942]
Regex (baseline)     Structural           0.7953  [0.6956, 0.8795]
CRF                  Spans Exact          0.8414  [0.7108, 0.9449]
CRF                  Spans Partial        0.8966  [0.7848, 0.9790]
CRF                  Structural           0.8914  [0.7865, 0.9759]
IndoBERT NER         Spans Exact          0.9747  [0.9348, 1.0000]
IndoBERT NER         Spans Partial        1.0000  [1.0000, 1.0000]
IndoBERT NER         Structural           0.9785  [0.9406, 1.0000]
